# Week 5 Solutions


## 1) Modified Gram-Schmidt (MGS)

We build a QR factorization of the design matrix $X$ using MGS, then solve the
least squares problem $\min_{\beta} \|X\beta - y\|_2$ via $X = QR$ and
$\beta = R^{-1} Q^T y$.


In [2]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(22)

# Parameters
n = 200  # number of observations
p = 20   # number of predictors

# Simulate predictor matrix X from standard normal distribution
X = np.random.randn(n, p)

# Simulate true coefficients beta (can be sparse or dense)
beta_true = np.random.randn(p)

# Simulate noise
epsilon = np.random.randn(n) * 0.5  # standard deviation of noise

# Generate response variable y
y = X @ beta_true + epsilon

print(X.shape)  # Check the shape of X


(200, 20)


In [3]:
def modified_gram_schmidt(A):
    """Return Q, R from the Modified Gram-Schmidt QR factorization."""
    A = A.astype(float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    V = A.copy()

    for i in range(n):
        R[i, i] = np.linalg.norm(V[:, i])
        if R[i, i] == 0:
            raise ValueError('Matrix has linearly dependent columns.')
        Q[:, i] = V[:, i] / R[i, i]
        for j in range(i + 1, n):
            R[i, j] = Q[:, i].T @ V[:, j]
            V[:, j] = V[:, j] - R[i, j] * Q[:, i]

    return Q, R

Q, R = modified_gram_schmidt(X)

# Solve least squares via QR
beta_mgs = np.linalg.solve(R, Q.T @ y)

# Compare with NumPy's least squares
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

print('||beta_mgs - beta_lstsq||_2 =', np.linalg.norm(beta_mgs - beta_lstsq))
print('Residual norm (MGS) =', np.linalg.norm(X @ beta_mgs - y))


||beta_mgs - beta_lstsq||_2 = 6.357784022188809e-15
Residual norm (MGS) = 6.441574010546481


**Example when (classical) Gram-Schmidt can fail**

If columns of $X$ are nearly linearly dependent, roundoff error destroys orthogonality
in the computed $Q$. Classical Gram-Schmidt is especially sensitive; MGS is better but
still suffers when the condition number is extremely large.


In [4]:
# Nearly dependent columns example
np.random.seed(7)
m = 50
v = np.random.randn(m)
A_bad = np.column_stack([v, v + 1e-12*np.random.randn(m), v - 1e-12*np.random.randn(m)])

def classical_gram_schmidt(A):
    A = A.astype(float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i].T @ v
            v = v - R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if R[j, j] == 0:
            raise ValueError('Matrix has linearly dependent columns.')
        Q[:, j] = v / R[j, j]
    return Q, R

Q_cgs, _ = classical_gram_schmidt(A_bad)
Q_mgs, _ = modified_gram_schmidt(A_bad)

orth_err_cgs = np.linalg.norm(Q_cgs.T @ Q_cgs - np.eye(Q_cgs.shape[1]))
orth_err_mgs = np.linalg.norm(Q_mgs.T @ Q_mgs - np.eye(Q_mgs.shape[1]))

print('Orthogonality error (CGS) =', orth_err_cgs)
print('Orthogonality error (MGS) =', orth_err_mgs)


Orthogonality error (CGS) = 0.00045205837470699
Orthogonality error (MGS) = 0.00045205837470699


## 2) The Rayleigh Quotient

For a (real) symmetric matrix $A$, the Rayleigh quotient is
$r(x) = \frac{x^T A x}{x^T x}$. Its stationary points occur at eigenvectors of $A$,
and the value equals the corresponding eigenvalue.


In [5]:
# Set random seed
np.random.seed(12)

# Step 1: Generate a random symmetric matrix A (e.g., a covariance-like matrix)
m = 5
A_random = np.random.randn(m, m)
A = A_random.T @ A_random  # ensures A is symmetric positive semi-definite

# Eigenvalues/eigenvectors (symmetric => use eigh)
eigvals, eigvecs = np.linalg.eigh(A)

print('Eigenvalues:', eigvals)

# Verify Rayleigh quotient on eigenvectors
rq = []
for i in range(m):
    x = eigvecs[:, i]
    rq_val = (x.T @ A @ x) / (x.T @ x)
    rq.append(rq_val)

print('Rayleigh quotients at eigenvectors:', rq)


Eigenvalues: [1.18324465e-02 1.43184850e+00 2.40179166e+00 3.57717012e+00
 1.98037784e+01]
Rayleigh quotients at eigenvectors: [np.float64(0.01183244654259883), np.float64(1.4318484950513084), np.float64(2.401791659704264), np.float64(3.577170115778478), np.float64(19.80377835793267)]


**If $A$ is not symmetric:**

The Rayleigh quotient $x^T A x / x^T x$ may be complex (or not real) and its
stationary points no longer align neatly with eigenvectors unless $A$ is normal
(e.g., $A^T A = A A^T$). In practice, for non-symmetric $A$ one typically uses
the symmetric part $(A + A^T)/2$ to get real-valued quotients, or applies
Rayleigh-quotient iteration with shifts while allowing complex arithmetic.
